# Exploring Geodatabases

Most GIS data in ArcGIS Pro is stored in geodatabases. In `arcpy-functions.ipynb`, you listed the files and workspaces in a folder, and `ListWorkspaces()` found the file geodatabases inside it. This notebook looks inside a geodatabase. You will list the feature datasets and feature classes it contains, filter them by name and geometry type, and describe what a feature class holds, so that a script can find and process the data in a geodatabase without typing each name.

🎯 What You'll Learn

* Set up a notebook to work in a file geodatabase
* Describe how a geodatabase organizes feature datasets and feature classes
* List the feature datasets in a geodatabase
* List feature classes at the root of a geodatabase and inside a feature dataset
* Filter feature classes by geometry type
* Combine a wildcard, a geometry type, and a feature dataset in one list
* Describe a feature class, including its geometry type and coordinate system

Work through each numbered section: a demo shows how something works, then a ✏️ **Practice** prompt below it gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

The **kernel** is the program that runs the Python code in this notebook. Before running any cells, make sure the notebook is using the ArcGIS Pro Python environment. An **environment** is a copy of Python plus the add-on tools installed with it.

Check the kernel name in the top-right corner of the notebook. It should read `arcgispro-py3`, and may include a Python version, for example:
`arcgispro-py3 (Python 3.13.13)`

If it shows a different name or says **Select Kernel**:

1. Click the kernel name (or **Select Kernel**) in the top-right corner
2. Select `arcgispro-py3`
3. If it is not listed, refer back to `jupyter-notebook-basics.ipynb` in `M1-getting-started-with-python\code-alongs`

---

### 1. Setup: ArcPy and the geodatabase workspace

Each notebook runs in its own kernel. Variables, imports, and settings from another notebook do not carry over, so every ArcPy notebook starts the same way that every ArcPy script does: import ArcPy, then set the environment settings.

⚠️ **Important**: If the next cell raises an exception, try these steps:

* Check that the kernel for this notebook is set to `arcgispro-py3`.
* Open ArcGIS Pro and sign in to the `uagis` organization account with your NetID and password so that ArcGIS Pro can assign you a Named User license.

If none of these steps fixes the exception, reach out to your instructor with a screenshot of the error message.

**DO NOT** proceed with this notebook until this exception is resolved.

In [ ]:
# Import the ArcPy site package
import arcpy

In `arcpy-functions.ipynb`, the workspace was a folder. A file geodatabase is also a workspace, so you can set `arcpy.env.workspace` to a geodatabase in the same way. Every list function in this notebook then searches inside that geodatabase.

This notebook uses the City of Austin file geodatabase, `Austin.gdb`, in the data folder. The next cell stores the full path to the geodatabase as a raw string and uses `arcpy.Exists()` to confirm that the path is correct before going further.

⚠️ **Important**: Update the following path to match the location of `Austin.gdb` on your computer.

In [ ]:
# Store the path to the Austin geodatabase. Update this path to match the
# location of Austin.gdb on your computer
austin_gdb = r"C:\UA\gist\Refresh_413_513_Fall2026\data\austin_data\Austin.gdb"
print(f"austin_gdb: {austin_gdb}")

# Confirm that the geodatabase exists before using it
print(f"Austin.gdb exists: {arcpy.Exists(austin_gdb)}")

⚠️ **Important**: If `Austin.gdb exists` prints `False`, fix the paths in the cell above before continuing. Every cell below depends on the workspace being set to the geodatabase.

In [ ]:
# Set the workspace to the geodatabase
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. What is inside a geodatabase

A **geodatabase** is a workspace that stores GIS data together with the rules for how that data is organized. A **file geodatabase** is stored on disk as a folder whose name ends in `.gdb`. The files inside that folder are managed by ArcGIS, and they do not match up one to one with the datasets you see in ArcGIS Pro. To find the datasets in a geodatabase, ArcPy provides list functions that understand how a geodatabase is organized.

A geodatabase can contain the following:

| Item | What it is |
| --- | --- |
| **Feature class** | A collection of features that share one geometry type, such as points, lines, or polygons, and the same attribute fields. A feature class is the geodatabase equivalent of a shapefile. |
| **Feature dataset** | A container inside a geodatabase that groups thematically related feature classes. All feature classes in a feature dataset share the same coordinate system. |
| **Standalone feature class** | A feature class stored at the **root** of the geodatabase, the top level, rather than inside a feature dataset. |
| **Table** | Attribute data with no geometry. |
| **Raster Dataset** | Data stored as a grid of cells, such as an elevation surface or an image. |

A partial view of `Austin.gdb`:

```
Austin.gdb
├── Administrative           (feature dataset)
│   ├── addresspts           (feature class)
│   ├── tracts               (feature class)
│   └── ...
├── Environmental            (feature dataset)
│   ├── watersheds           (feature class)
│   └── ...
├── Parks                    (feature dataset)
├── Transportation           (feature dataset)
├── standalone feature classes
├── tables
└── rasters
```

For more information on Geodatabases, refer to [Esri Help Documentation](https://doc.esri.com/en/arcgis-pro/latest/help/data/geodatabases/overview/what-is-a-geodatabase-.html).
This notebook focuses on feature datasets and feature classes. Tables and rasters have list functions of their own.

---

### 3. Listing feature datasets

`ListWorkspaces()` found the geodatabases inside a folder. One level down, inside a geodatabase, `arcpy.ListDatasets()` returns a list of the names of the datasets in the workspace. A **feature dataset** is any collection of thematically similar data with the same coordinate system that ArcGIS stores and manages as one unit. Feature datasets are one kind of dataset, and a geodatabase can hold other kinds as well.

The optional `feature_type` parameter limits the results to one kind of dataset. `feature_type="Feature"` returns only feature datasets. This works the same way as the `workspace_type` parameter of `ListWorkspaces()`: `feature_type` is the second optional parameter, so the clearest way to set it is with a keyword argument.

In [ ]:
# Example 1 - List all datasets in the geodatabase

# We will use the Austin.gdb set in arcpy.env.workspace
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

list_of_datasets = arcpy.ListDatasets()

# ListDatasets returns a list object
print(f"type(list_of_datasets): {type(list_of_datasets)}")

# Use a for loop to print each dataset
print(f"Datasets in {arcpy.env.workspace}:")
for i, one_dataset in enumerate(list_of_datasets, start=1):
    print(f"\t{i}. {one_dataset}")

In [ ]:
# Example 2 - List only feature datasets
list_of_feature_datasets = arcpy.ListDatasets(feature_type="Feature")

# Use a for loop to print each feature dataset
print(f"Feature datasets in {arcpy.env.workspace}:")
for i, one_dataset in enumerate(list_of_feature_datasets, start=1):
    print(f"\t{i}. {one_dataset}")

# Compare this output with the output of Example 1

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Listing feature classes: root vs. feature dataset

`arcpy.ListFeatureClasses()` returns a list of the names of the feature classes in the workspace. Like `ListFiles()` and `ListWorkspaces()`, it searches only the workspace itself. When the workspace is a geodatabase, that means the root of the geodatabase.

🔍 [VERIFY: confirm on Austin.gdb that the root list below excludes feature classes inside feature datasets before keeping this sentence] Called with no arguments in a geodatabase, `ListFeatureClasses()` returns only the standalone feature classes. The feature classes inside feature datasets are not included.

There are two ways to list the feature classes inside a feature dataset:

* Use the optional `feature_dataset` parameter to name the feature dataset. It is the third optional parameter, so set it with a keyword argument: `feature_dataset="Administrative"`.
* Set the workspace to the feature dataset itself. A feature dataset is a workspace inside the geodatabase, and its path is the geodatabase path followed by the feature dataset name.

In [ ]:
# Example 1 - List the standalone feature classes at the root

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

list_of_root_fcs = arcpy.ListFeatureClasses()
print(f"Number of standalone feature classes: {len(list_of_root_fcs)}")
print("Standalone feature classes:")
for one_fc in list_of_root_fcs:
    print(f"\t{one_fc}")

In [ ]:
# Example 2 - List the feature classes in a feature dataset by name

list_of_admin_fcs = arcpy.ListFeatureClasses(
    feature_dataset="Administrative"
)
print(f"Number of feature classes: {len(list_of_admin_fcs)}")
print("Feature classes in Administrative:")
for one_fc in list_of_admin_fcs:
    print(f"\t{one_fc}")

# Compare this output with the output of Example 1

In [ ]:
# Example 3 - Set the workspace to a feature dataset

# A feature dataset path is the geodatabase path plus the dataset name
parks_dataset = rf"{austin_gdb}\Parks"
arcpy.env.workspace = parks_dataset
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# No feature_dataset argument is needed: the workspace is the dataset
list_of_parks_fcs = arcpy.ListFeatureClasses()
print("Feature classes in Parks:")
for one_fc in list_of_parks_fcs:
    print(f"\t{one_fc}")

# Set the workspace back to the geodatabase for the rest of this notebook
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 4a] Set the workspace to the Austin geodatabase, and print the workspace.
# Use arcpy.ListFeatureClasses() to store a list of the standalone feature
# classes. Print the number of feature classes in the list.


# 4b] Use arcpy.ListFeatureClasses() with the feature_dataset parameter to
# store a list of the feature classes in the feature dataset you chose in
# 3d. Use a for loop to print the name of each feature class.


# 4c] Use an rf-string to create a variable that stores the path of the
# feature dataset you chose in 3d. Set the workspace to this path, and
# print the workspace.


# 4d] Use arcpy.ListFeatureClasses() with no arguments to store a list of
# the feature classes in the workspace from 4c. Use a for loop to print the
# name of each feature class, and compare the output with 4b.


# 4e] Set the workspace back to the Austin geodatabase, and print the
# workspace.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Filtering by geometry type

In Section 3, the `feature_type` parameter of `ListDatasets()` kept only feature datasets. `ListFeatureClasses()` has a parameter with the same name, but here it filters feature classes by **geometry type**, the kind of shape that every feature in a feature class has. The most common values are:

| `feature_type` value | Returns |
| --- | --- |
| `"Point"` | Point feature classes, such as address points |
| `"Polyline"` | Line feature classes, such as streets |
| `"Polygon"` | Polygon feature classes, such as census tracts |

🔍 [VERIFY: confirm "Polyline" returns the line feature classes in Transportation] `feature_type` is the second optional parameter and `feature_dataset` is the third, so set both with keyword arguments. Their order does not matter when you name them.

In [ ]:
# Example 1 - List the polygon feature classes in a feature dataset

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

admin_polygons = arcpy.ListFeatureClasses(
    feature_type="Polygon", feature_dataset="Administrative"
)
print("Polygon feature classes in Administrative:")
for one_fc in admin_polygons:
    print(f"\t{one_fc}")

In [ ]:
# Example 2 - List the line feature classes in a feature dataset

transportation_lines = arcpy.ListFeatureClasses(
    feature_type="Polyline", feature_dataset="Transportation"
)
print("Line feature classes in Transportation:")
for one_fc in transportation_lines:
    print(f"\t{one_fc}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Choose one geometry type: point, line, or polygon. Use
# arcpy.ListFeatureClasses() with the feature_type and feature_dataset
# parameters to store a list of the feature classes of that type in the
# feature dataset you chose in 3d. Use a for loop to print each name.


# 5b] Use arcpy.ListFeatureClasses() with only the feature_type parameter
# to store a list of the standalone feature classes of the geometry type
# you chose in 5a. Use a for loop to print each name.


# 5c] Repeat 5a with a different geometry type.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. Combining a wildcard, a geometry type, and a feature dataset

In `arcpy-functions.ipynb`, you used wildcards with `ListFiles()` to filter file names, such as `"*.shp"` for every name that ends with `.shp`. The first optional parameter of `ListFeatureClasses()` is also a wildcard, `wild_card`, and it filters feature class names with the same `*` pattern. Because `wild_card` is the first parameter, you can pass it without a keyword.

The wildcard, the geometry type, and the feature dataset can be used together. Each filter narrows the list further, so the result contains only the feature classes that match all of them.

In [ ]:
# Example 1 - Filter by name in a feature dataset

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Feature classes in Parks with "ball" anywhere in the name
ball_fcs = arcpy.ListFeatureClasses("*ball*", feature_dataset="Parks")
print("Feature classes in Parks with 'ball' in the name:")
for one_fc in ball_fcs:
    print(f"\t{one_fc}")

In [ ]:
# Example 2 - Filter by name and geometry type in a feature dataset

ball_points = arcpy.ListFeatureClasses(
    "*ball*", feature_type="Point", feature_dataset="Parks"
)
print("Point feature classes in Parks with 'ball' in the name:")
for one_fc in ball_points:
    print(f"\t{one_fc}")

# Compare this output with the output of Example 1

Older code, including a lot of AI-generated code, passes every argument by position and uses an empty string `""` to skip a parameter, such as `arcpy.ListFeatureClasses("", "Point", "Parks")`. It returns the same result. Recognize this syntax when you see it, but you do not need to write it.

In [ ]:
# Older syntax: arguments by position
# arcpy.ListFeatureClasses("*ball*", "Point", "Parks")

# OR

# Older syntax: an empty string in place of the wildcard
# arcpy.ListFeatureClasses("", "Point", "Parks")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 6a] Choose a letter or a short word that appears in the name of at least
# one feature class in the feature dataset you chose in 3d. Use
# arcpy.ListFeatureClasses() with a wildcard and the feature_dataset
# parameter to store a list of the feature classes whose names contain it.
# Use a for loop to print each name.


# 6b] Add the feature_type parameter to your list from 6a to keep only one
# geometry type. Use a for loop to print each name.


# 6c] Repeat 6a, but change the letter or word in your wildcard from
# lowercase to uppercase, or from uppercase to lowercase. Use a for loop to
# print each name. In a comment, write whether the results changed.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 7. Describing a feature class

The list functions return names. A name tells you what a feature class is called, but not what it contains. `arcpy.Describe()` is an ArcPy function that takes a dataset name or a full path and returns a **Describe object**, an object whose properties describe the dataset.

The properties of a Describe object depend on the kind of data it describes. For a feature class, three useful properties are:

* `dataType`: the kind of dataset, such as `FeatureClass`
* `shapeType`: the geometry type, such as `Point`, `Polyline`, or `Polygon`
* `spatialReference`: the coordinate system of the feature class, as a `SpatialReference` object

`spatialReference` returns an object from the same `SpatialReference` class you used in `arcpy-site-package.ipynb`. There, you created the object from a factory code. Here, ArcPy creates it from the data, and you read its properties, such as `name` and `factoryCode`, in the same way.

In [ ]:
# Example 1 - Describe one standalone feature class

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# We will use variable list_of_root_fcs from Section 4
one_fc = list_of_root_fcs[0]
print(f"one_fc: {one_fc}")

fc_desc = arcpy.Describe(one_fc)
print(f"type(fc_desc): {type(fc_desc)}")
print(f"Data type: {fc_desc.dataType}")
print(f"Shape type: {fc_desc.shapeType}")

# spatialReference stores a SpatialReference object
fc_sr = fc_desc.spatialReference
print(f"type(fc_sr): {type(fc_sr)}")
print(f"Coordinate system: {fc_sr.name}")
print(f"Factory code: {fc_sr.factoryCode}")

🔍 [VERIFY: confirm that arcpy.Describe() finds a feature class inside a feature dataset by name alone when the workspace is the geodatabase. If it does not, Example 2 needs the path rf"{austin_gdb}\Administrative\{one_fc}" instead] Because `Describe()` works on one dataset at a time, a for loop over a list of feature class names describes every feature class in the list.

In [ ]:
# Example 2 - Describe every feature class in a list

# We will use variable admin_polygons from Section 5
print("Polygon feature classes in Administrative:")
for one_fc in admin_polygons:
    fc_desc = arcpy.Describe(one_fc)
    fc_sr = fc_desc.spatialReference
    print(f"\t{one_fc} | {fc_desc.shapeType} | {fc_sr.name}")

# The shape type confirms the feature_type filter from Section 5

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 7a] Choose one feature class from your list in 4b. Use arcpy.Describe()
# to store its Describe object in a variable. Print its data type, shape
# type, coordinate system name, and factory code, and label each value.


# 7b] Use a for loop on your list from 4b. For each feature class, use
# arcpy.Describe() to print its name, shape type, and coordinate system
# name on one line.


# 7c] In a comment, write whether every feature class from 7b has the same
# coordinate system, and explain why, based on what a feature dataset is.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*